# Revisión capa BRONCE (datos crudos)

Inventario de los datos originales de la NYC TLC tal cual se descargaron.
Esta capa es **inmutable**: nada del pipeline la modifica.

Se espera: 144 archivos (4 categorías × 12 meses × 3 años: 2023–2025)
+ zone lookup + auditoría.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Inventario por categoría

In [2]:
resumen = []
for cat in CATS:
    d = DATA / "bronze" / cat
    archivos = sorted(d.glob("*.parquet"))
    resumen.append({
        "categoria": cat,
        "archivos": len(archivos),
        "filas": parquet_rows(d),
        "peso_mb": dir_mb(d),
    })
inv = pl.DataFrame(resumen)
print(f"TOTAL bronce: {inv['filas'].sum():,} filas | {inv['peso_mb'].sum():,.0f} MB")
inv

TOTAL bronce: 904,327,862 filas | 19,473 MB


categoria,archivos,filas,peso_mb
str,i64,i64,f64
"""green""",36,2038653,46.75
"""yellow""",36,128202548,2058.71
"""fhv""",36,58536509,624.43
"""fhvhv""",36,715550152,16742.82


## Detalle mensual (filas por archivo, 36 meses)

In [3]:
detalle = []
for cat in CATS:
    for y, m in MESES:
        f = DATA / "bronze" / cat / f"{y}-{m:02d}.parquet"
        detalle.append({"categoria": cat, "mes": f"{y}-{m:02d}", "filas": parquet_rows(f)})
(pl.DataFrame(detalle)
   .pivot(values="filas", index="mes", on="categoria")
   .sort("mes"))

mes,green,yellow,fhv,fhvhv
str,i64,i64,i64,i64
"""2023-01""",68211,3066766,1114320,18479031
"""2023-02""",64809,2913955,1110797,17960971
"""2023-03""",72044,3403766,1328242,20413539
"""2023-04""",65392,3288250,1246479,19144903
"""2023-05""",69174,3513649,1385826,19847676
"""2023-06""",65550,3307234,1219445,19366619
"""2023-07""",61343,2907108,1370843,19132131
"""2023-08""",60649,2824209,1440352,18322150
"""2023-09""",65471,2846722,1293303,19851123


## Tabla de zonas (zone lookup)

In [4]:
zonas = pl.read_parquet(DATA / "bronze" / "zone-lookup" / "zone-lookup-table.parquet")
print(f"{zonas.height} zonas | boroughs: {sorted(zonas['Borough'].unique().to_list())}")
zonas.head(5)

265 zonas | boroughs: ['Bronx', 'Brooklyn', 'EWR', 'Manhattan', 'N/A', 'Queens', 'Staten Island', 'Unknown']


LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


## Auditoría de descargas

In [5]:
audit_b = pl.read_parquet(DATA / "bronze" / "audit.parquet")
print(f"{audit_b.height} registros de descarga | filas totales auditadas: {audit_b['rowcount'].sum():,}")
audit_b.select("name", "rowcount", "bytecount", "end_timestamp").tail(10)

149 registros de descarga | filas totales auditadas: 904,329,187


name,rowcount,bytecount,end_timestamp
str,i64,i64,str
"""fhvhv_2024-05""",20704538,498575918,"""2026-07-02T17:50:19.591799"""
"""fhvhv_2024-06""",20123226,484417715,"""2026-07-02T17:50:36.486691"""
"""fhvhv_2024-07""",19182934,468437522,"""2026-07-02T17:50:52.502833"""
"""fhvhv_2024-08""",19128392,465991646,"""2026-07-02T17:51:08.377125"""
"""fhvhv_2024-09""",19209788,466470118,"""2026-07-02T17:51:24.758035"""
"""fhvhv_2024-10""",20028282,485102359,"""2026-07-02T17:51:41.248056"""
"""fhvhv_2024-11""",19987533,480228052,"""2026-07-02T17:51:57.458774"""
"""fhvhv_2024-12""",21068851,507534115,"""2026-07-02T17:52:14.659591"""
"""zone-lookup-table""",265,4968,"""2026-07-02T18:41:00.294208"""
